In [56]:
from statistics import LinearRegression

import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
#from sklearn.cross_validation import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import re


In [57]:
# Data import for the use in th notebook
spam_data = pd.read_csv(
    "../dataset/sms_scam_detection_dataset_merged_with_lang.csv",
    encoding="utf-8",
    low_memory=False
)
print(spam_data.columns)


Index(['label', 'text', 'URL', 'EMAIL', 'PHONE', 'lang', 'Unnamed: 6',
       'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10'],
      dtype='str')


In [58]:
# Exploration of the dataset
spam_data.head()

,label,text,URL,EMAIL,PHONE,lang,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
1,ham,What's up? Do you want me to come online? If y...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
2,ham,So u workin overtime nigpun?,No,No,No,en,NaN,NaN,NaN,NaN,NaN
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,en,NaN,NaN,NaN,NaN,NaN
4,spam,Please Stay At Home. To encourage the notion o...,No,No,No,en,NaN,NaN,NaN,NaN,NaN


In [62]:
# As the column label contain only Ham and spam , we are gonna replace them by 1 and 0
spam_data= spam_data.replace(['ham','spam','Spam','Ham'],[0,1,1,0])


In [5]:
# Some column have NAN value
# We must suppress all the NAN columns
spam_data = spam_data.drop(columns=["Unnamed: 7","Unnamed: 8","Unnamed: 9","Unnamed: 10"])
spam_data = spam_data.drop(columns=["Unnamed: 6"])

In [6]:
spam_data = spam_data.rename(columns={
    "label": "Label",
    "text": "Text",
    "URL": "Url",
    "EMAIL": "Email",
    "PHONE": "Phone",
    "lang": "Language"
})
spam_data.head()

,Label,Text,Url,Email,Phone,Language
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en
1,0,What's up? Do you want me to come online? If y...,No,No,No,en
2,0,So u workin overtime nigpun?,No,No,No,en
3,0,"Also sir, i sent you an email about how to log...",No,No,No,en
4,1,Please Stay At Home. To encourage the notion o...,No,No,No,en


In [7]:
# Let's treat the others columns
# column URL
print("Unique Value for the column URL")
spam_data["Url"].value_counts()


Unique Value for the column URL


Url
No     5744
yes     205
Name: count, dtype: int64

In [8]:
spam_data['Url'] = spam_data['Url'].replace({'Yes': 1, 'No': 0})

In [9]:
print("Unique Value for the column EMAIL")
spam_data['Email'].value_counts()



Unique Value for the column EMAIL


Email
No     5930
yes      19
Name: count, dtype: int64

In [10]:
spam_data['Email']=spam_data['Email'].replace({'Yes':1,'No':0})

In [11]:
print("Unique Value for the column PHONE")
spam_data["Phone"].value_counts()

Unique Value for the column PHONE


Phone
No     5115
yes     834
Name: count, dtype: int64

In [12]:
spam_data['Phone']=spam_data['Phone'].replace({'Yes':1,'No':0})

In [13]:
print("Unique Value for the column lang")
spam_data['Language'].value_counts()

Unique Value for the column lang


Language
en         5850
hindi      5129
french     5108
german     5108
english     275
und          13
hi            8
ml            8
nl            7
no            6
kn            5
ro            5
kha           4
ta            4
ru            3
bn            3
es            3
te            3
ar            2
pt            2
lb            2
zh            2
it            1
af            1
fil           1
ms            1
sv            1
fr            1
ku            1
ht            1
aa            1
sw            1
ny            1
tr            1
hr            1
vi            1
pl            1
kri           1
gu            1
fy            1
sm            1
Name: count, dtype: int64

In [14]:
# The column for languages contains the language according the ISO 931 format
# We should try to normalize or simplify the use of this column
mapping = {
    # Déjà sous forme de codes ISO 639-1
    "en": "english",
    "fr": "french",
    "de": "german",
    "hi": "hindi",
    "ml": "malayalam",
    "nl": "dutch",
    "no": "norwegian",
    "kn": "kannada",
    "ro": "romanian",
    "ta": "tamil",
    "ru": "russian",
    "bn": "bengali",
    "es": "spanish",
    "te": "telugu",
    "ar": "arabic",
    "pt": "portuguese",
    "zh": "chinese",
    "it": "italian",
    "af": "afrikaans",
    "ms": "malay",
    "sv": "swedish",
    "ku": "kurdish",
    "ht": "haitian creole",
    "sw": "swahili",
    "ny": "chichewa",
    "tr": "turkish",
    "hr": "croatian",
    "vi": "vietnamese",
    "pl": "polish",
    "gu": "gujarati",
    "fy": "western frisian",
    "sm": "samoan",
    "aa": "afar",

    # Codes ISO 639-3
    "fil": "filipino",
    "kha": "khasi",
    "kri": "krio",

    # Cas particulier
    "und": "unknown"
}

In [15]:
spam_data['Language'] = spam_data['Language'].replace(mapping)

In [16]:
spam_data['Language'].value_counts()

Language
english            6125
hindi              5137
french             5109
german             5108
unknown              13
malayalam             8
dutch                 7
norwegian             6
kannada               5
romanian              5
khasi                 4
tamil                 4
russian               3
bengali               3
spanish               3
telugu                3
arabic                2
portuguese            2
lb                    2
chinese               2
italian               1
afrikaans             1
filipino              1
malay                 1
swedish               1
kurdish               1
haitian creole        1
afar                  1
swahili               1
chichewa              1
turkish               1
croatian              1
vietnamese            1
polish                1
krio                  1
gujarati              1
western frisian       1
samoan                1
Name: count, dtype: int64

In [17]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english
1,0,What's up? Do you want me to come online? If y...,0,0,0,english
2,0,So u workin overtime nigpun?,0,0,0,english
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english


In [18]:
spam_data.dtypes

Label       object
Text        object
Url         object
Email       object
Phone       object
Language       str
dtype: object

In [19]:
# Let's treat the column Test with NLP
# 1 - Count the number of word in each text
spam_data['Word_Count'] = 0
for i in np.arange(0, len(spam_data.Text)):
    spam_data.loc[i, 'Word_Count'] = len(str(spam_data.loc[i, 'Text']).split())

In [20]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [21]:
corpus = []
ps = PorterStemmer()

In [22]:
# Some original messages
print (spam_data["Text"][0])
spam_data["Text"][1]

Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matured 8. Stylish 9. Simple Pls reply..


"What's up? Do you want me to come online? If you are free we can talk sometime�"

In [23]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [24]:
# Translator to translate all text in english to normalize the dataset
from deep_translator import  GoogleTranslator
def translate_sentence(sentence):
    traducteur = GoogleTranslator(source="auto",destination="en")
    translated_sentence = traducteur.translate(sentence)
    return translated_sentence

In [25]:
# Let's translate all languages others than english
spam_data = spam_data[spam_data["Language"] == "english"]
# to suppress the old index that are NAN now
spam_data = spam_data.reset_index(drop=True)

In [26]:
# Let's print the new dataset
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [27]:

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jemnis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
corpus = []

for i in range(len(spam_data)):

    msg = str(spam_data['Text'][i])

    # Regex
    msg = re.sub(r'\b[\w\-.]+?@\w+?\.\w{2,4}\b', 'emailaddr', msg)
    msg = re.sub(r'(http[s]?\S+)|(\w+\.[A-Za-z]{2,4}\S*)', 'httpaddr', msg)
    msg = re.sub(r'£|\$', 'moneysymb', msg)
    msg = re.sub(r'\b(\+\d{1,2}\s)?\d?[\-(.]?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b', 'phonenumbr', msg)
    msg = re.sub(r'\d+(\.\d+)?', 'numbr', msg)

    # Remove punctuation
    msg = re.sub(r'[^\w\d\s]', ' ', msg)

    # Lowercase
    msg = msg.lower()

    # Tokenize
    words = msg.split()

    # Stopwords + stemming (anglais seulement)
    words = [
        ps.stem(word)
        for word in words
        if word not in set(stopwords.words('english'))
    ]

    msg = ' '.join(words)

    corpus.append(msg)

In [29]:
cv = CountVectorizer(max_features=3000)
x = cv.fit_transform(corpus)

x_df = pd.DataFrame(
    x.toarray(),
    columns=cv.get_feature_names_out()
)

In [30]:
# Let's split our dataset
X = spam_data.drop(columns=['Text', 'Label', 'Language','Url','Email','Phone']).reset_index(drop=True)

X = pd.concat([X, x_df], axis=1)

X.head()


,Word_Count,aah,aathi,aberdeen,abi,abil,abiola,abj,abl,abt,...,yogasana,yor,your,yr,yummi,yun,yunni,yuo,yup,zed
0,26,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,17,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,33,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [31]:
# y for the output value
y= spam_data['Label']
y= y.astype(int)

In [32]:
from sklearn.metrics import  accuracy_score , r2_score , recall_score , f1_score
from sklearn.tree import DecisionTreeClassifier
from  sklearn.model_selection import  train_test_split
print(X.select_dtypes(include='object').columns)
print(type(y))
print(y.dtype)
print(y.head())
print(y.unique())

Index([], dtype='str')
<class 'pandas.Series'>
int64
0    0
1    0
2    0
3    0
4    1
Name: Label, dtype: int64
[0 1]


In [33]:
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

In [34]:

model = DecisionTreeClassifier()

model.fit(X_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [35]:
model.score(X_val,y_val)

0.9616326530612245

In [36]:
from sklearn.metrics import  mean_squared_error , mean_absolute_error
mean_squared_error(y_val,model.predict(X_val))

0.03836734693877551

In [37]:
model.predict(X_val)

array([1, 0, 0, ..., 0, 1, 0], shape=(1225,))

In [38]:
predictions = model.predict(X_val)

resultats = pd.DataFrame({
    "Valeur réelle": y_val.reset_index(drop=True),
    "Prédiction": predictions
})

print(resultats.head())

   Valeur réelle  Prédiction
0              1           1
1              0           0
2              0           0
3              0           0
4              0           0


In [39]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       0.97      0.98      0.98       977
           1       0.92      0.88      0.90       248

    accuracy                           0.96      1225
   macro avg       0.95      0.93      0.94      1225
weighted avg       0.96      0.96      0.96      1225

[[959  18]
 [ 29 219]]


In [40]:
# Analysons la matrice de confusion
print(confusion_matrix(y_val, y_pred))

[[959  18]
 [ 29 219]]


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , r2_score , recall_score , f1_score
from sklearn.svm import LinearSVC
# Testons d'autres modèles
model_one = LogisticRegression(class_weight="balanced")
model_two = LinearSVC(class_weight="balanced")


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []
trained_models = {}

models = {
    "Logistic Regression": LogisticRegression(),
    "Linear SVM": LinearSVC(),
    "Linear SVM balanced": LinearSVC(class_weight="balanced"),
    "Logistic Regression balanced": LogisticRegression(class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier()
}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1-score": f1_score(y_val, y_pred)
    })

    # sauvegarde du modèle entraîné
    trained_models[name] = model


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1-score",
    ascending=False
)

results_df

C:\Users\jemnis\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\jemnis\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://sci

,Model,Accuracy,Precision,Recall,F1-score
3,Logistic Regression balanced,0.982041,0.963115,0.947581,0.955285
2,Linear SVM balanced,0.978776,0.970339,0.923387,0.946281
0,Logistic Regression,0.977959,0.970213,0.919355,0.944099
1,Linear SVM,0.976327,0.965957,0.915323,0.939959
4,Decision Tree,0.962449,0.931624,0.879032,0.904564


In [51]:
print(X_train.columns)

Index(['Word_Count', 'aah', 'aathi', 'aberdeen', 'abi', 'abil', 'abiola',
       'abj', 'abl', 'abt',
       ...
       'yogasana', 'yor', 'your', 'yr', 'yummi', 'yun', 'yunni', 'yuo', 'yup',
       'zed'],
      dtype='str', length=3001)


In [54]:
# le modèle le plus performant est Logisitic Regression Balanced

best_model_name = results_df.iloc[0]["Model"]

best_model = trained_models[best_model_name]
new_message = ["Congratulations you won a free prize"]

# transformation texte
new_message_vector = cv.transform(new_message)

# conversion dataframe
new_message_df = pd.DataFrame(
    new_message_vector.toarray(),
    columns=cv.get_feature_names_out()
)
# ajout Word_Count
new_message_df["Word_Count"] = len(new_message[0].split())
new_message_df = new_message_df[X_train.columns]
print(new_message_df.columns)
prediction = best_model.predict(new_message_df)
print(prediction)



Index(['Word_Count', 'aah', 'aathi', 'aberdeen', 'abi', 'abil', 'abiola',
       'abj', 'abl', 'abt',
       ...
       'yogasana', 'yor', 'your', 'yr', 'yummi', 'yun', 'yunni', 'yuo', 'yup',
       'zed'],
      dtype='str', length=3001)
[0]


In [55]:
import joblib

# modèle
joblib.dump(best_model, "../models/spam_model.pkl")

# vectorizer
joblib.dump(cv, "../models/count_vectorizer.pkl")

# ordre des features
joblib.dump(X_train.columns, "../models/features_columns.pkl")

['features_columns.pkl']